# COGS 108 - EDA Checkpoint

## Authors

- Minghao Xu: Project administration, Methodology, Analysis, Experimental investigation
- Jerry Chen: Analysis, Software, Data curation, Methodology 
- Eli Liang:  Visualization, Conceptualization, Writing - review & editing
- William Wu: Analysis, Background research, Writing - original draft
- Weder Qin: Software, Visualization, Writing - review & editing

# Research Question

To what degree does the court surface (Clay, Grass, Hard) affect the first-serve and second-serve win percentage in men’s singles matches(All Matches) at the 2024 Grand Slam tournaments?



## Background and Prior Work

Court surface can change how much advantage a server gets because bounce height, skid, and pace affect the returner's reaction time and how quickly rallies "reset" after the serve. In Grand Slams, this creates a natural comparison across hard (Australian Open / US Open), clay (French Open), and grass (Wimbledon) to test whether first-serve win% and second-serve win% differ by surface in 2024 elite matches.

Prior work motivates our focus on first- and second-serve point outcomes. A widely shared modeling project shows that first-serve points won and second-serve points won are among the most informative serving statistics for predicting match outcomes, supporting their use as core response variables in our study.<a name="cite_ref-1"></a>[<sup>1</sup>](#cite_note-1) Meanwhile, serve-focused research using Australian Open data shows that outcomes like aces depend on multiple serve attributes (not just raw speed), reinforcing why we measure overall point win% rather than relying on aces alone.<a name="cite_ref-2"></a>[<sup>2</sup>](#cite_note-2) Consistent with that, tennis commentary analyses caution that aces are only one small component of winning and recommend broader serve-point measures as better indicators of serve dominance.<a name="cite_ref-3"></a>[<sup>3</sup>](#cite_note-3)

Building on these ideas, our project will estimate how much surface (clay/grass/hard) shifts first- and second-serve win% using match-level Grand Slam data from the 2024 season.<a name="cite_ref-4"></a>[<sup>4</sup>](#cite_note-4)

1. <a name="cite_note-1"></a> [^](#cite_ref-1) Kokta, M. (Medium). *Predicting ATP Tennis Match Outcomes Using Serving Statistics.* https://medium.com/swlh/predicting-atp-tennis-match-outcomes-using-serving-statistics-cde03d99f410  
2. <a name="cite_note-2"></a> [^](#cite_ref-2) Whiteside, D., et al. (2017). *Spatial characteristics of professional tennis serves with implications for serving aces: A machine learning approach.* *Journal of Sports Sciences.* https://www.tandfonline.com/doi/full/10.1080/02640414.2016.1183805  
3. <a name="cite_note-3"></a> [^](#cite_ref-3) TalkingTennis.net (Jan 24, 2025). *How important are aces in winning tennis matches?* https://talkingtennis.net/blog-posts/how-important-are-aces-in-winning-tennis-matches  
4. <a name="cite_note-4"></a> [^](#cite_ref-4) Sackmann, J. (ongoing). *ATP match-level data.* https://github.com/JeffSackmann/tennis_atp

# Hypothesis


We predict that court surface influences serve win percentages, with the highest first-serve win percentages occurring on grass and the lowest on clay, while second-serve win percentages will remain relatively stable across all surfaces. Since first serves usually aim for higher speed while second serves usually aim for stability, the court surface likely has a more pronounced influence on first serves compared to second serves. The Court Pace Ratings (CPR) are largest on grass courts and lowest on clay courts, so grass courts are expected to have more impact on first-serve winning rate.

## Data

### Data overview

- **Dataset #1**
  - **Dataset Name:** ATP Men's Tennis Match Results + Match Statistics (match-level dataset)
  - **Link to the dataset:** *https://github.com/JeffSackmann/tennis_atp/blob/master/atp_matches_2024.csv*
  - **Number of observations:** 3,076 match observations
  - **Number of variables:**  49 columns covering tournament, players, outcomes, and stats 
  - **Description of the variables most relevant to this project:**
    - **Tournament information:** `tourney_name`, `surface`, `tourney_level`, `tourney_date`  
      Identifies where/when the match happened and key context like **surface** (Hard/Clay/Grass) and tournament tier.
    - **Player information:** `winner_name`, `loser_name`, `winner_hand`, `loser_hand`, `winner_ht`, `loser_ht`, `winner_age`, `loser_age`  
      Demographics/attributes used to compare players and control for factors like age/height/handedness.
    - **Match outcome:** `score`, `round`, `minutes`, `best_of`  
      Outcome and competitiveness proxies (round reached, duration, format best-of-3 vs best-of-5).
    - **Serve statistics:** `w_ace`, `l_ace`, `w_df`, `l_df`, `w_svpt`, `l_svpt`, `w_1stIn`, `l_1stIn`  
      Core performance indicators (aces, double faults, service volume, first-serve success).
    - **Rankings:** `winner_rank`, `loser_rank`, `winner_rank_points`, `loser_rank_points`  
      Pre-match strength indicators to study expectations, upsets, and performance vs rank.
  - **Descriptions of any shortcomings this dataset has with respect to the project:**
    - **Missing data (~7% overall):** especially **seed** (missing for many matches) and **entry type**, plus some missing match statistics and duration.
    - **Incomplete matches:** retirements (RET) and walkovers (W/O) make some match stats unreliable or absent; these matches can bias analyses of performance metrics like aces/DFs/minutes.
    - **Missing duration for a non-trivial subset:** some matches lack `minutes`, which limits analyses involving match length/competitiveness.
    - **Limited context:** no injury info, weather, court-speed variation within surfaces, motivation effects, etc., so interpretation is limited to what's captured in match stats.
    - **Potential minor inconsistencies:** some static attributes (height/age) occasionally missing or inconsistently recorded across events.

**Combining datasets:** This project uses a single dataset, so no merging is required.

In [1]:
# Run this code every time when you're actively developing modules in .py files.  It's not needed if you aren't making modules
#
## this code is necessary for making sure that any modules we load are updated here 
## when their source code .py files are modified

%load_ext autoreload
%autoreload 2

In [ ]:
# Setup code -- Run only once after cloning!!!
#
# this code downloads the data from its source to the `data/00-raw/` directory
# if the data hasn't updated you don't need to do this again!

# if you don't already have these packages (you should!) uncomment this line
# %pip install requests tqdm

import sys
sys.path.append('./modules') # this tells python where to look for modules to import

import get_data # this is where we get the function we need to download data

# replace the urls and filenames in this list with your actual datafiles
# yes you can use Google drive share links or whatever
# format is a list of dictionaries; 
# each dict has keys of 
#   'url' where the resource is located
#   'filename' for the local filename where it will be stored 
datafiles = [
    {
        "url": "https://raw.githubusercontent.com/JeffSackmann/tennis_atp/master/atp_matches_2024.csv",
        "filename": "atp_matches_2024.csv",
    }
]

get_data.get_raw(datafiles, destination_directory="data/00-raw/")

### ATP Men’s Tennis Matches – 2024 Season 

The ATP Matches 2024 dataset contains detailed match-level information for all ATP men’s professional tennis matches played between January 1 and December 31, 2024. The data comes from Jeff Sackmann’s tennis_atp repository and includes 3,076 matches and 49 variables. Each row represents a single tennis match and each column represents one variable, meaning the dataset is already in tidy format.

The dataset includes several important categories of variables. Tournament information includes surface (Hard, Clay, Grass), tournament level (Grand Slam, Masters 1000, ATP 250/500), and date. Player information includes winner and loser names, handedness, height (cm), and age (years). Match outcome variables include score, round, best_of (3 or 5 sets), and match duration in minutes.

Several performance metrics are particularly important:

Aces (w_ace, l_ace): Number of serves that result in an immediate point. Measured as a count per match. Professional players typically average 5–15 aces per match. Higher ace counts generally indicate stronger serving performance.

Double Faults (w_df, l_df): Number of missed second serves. Measured as a count per match. Professionals usually aim to stay under 3–5 per match since double faults directly give points to the opponent.

Match Duration (minutes): Measured in minutes. Best-of-three matches typically last 90–150 minutes; best-of-five matches may exceed 3–5 hours.

ATP Ranking (winner_rank, loser_rank): Numerical ranking (1 = best). Rankings are based on a rolling 52-week point system. Lower numbers indicate stronger players.

Dataset Concerns:

Approximately 7% of all values are missing. Missingness is not random. It is strongly associated with incomplete matches (retirements and walkovers). There are 101 matches marked as walkover (W/O) or retirement (RET), and 238 matches missing duration data. Around 60 matches are missing serve statistics such as aces and double faults. These missing values are systematic and expected because incomplete matches do not generate full statistics.

There are also extreme match duration values (e.g., 0 minutes for walkovers and matches exceeding 220 minutes), but these represent legitimate real-world scenarios and should not automatically be removed.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("data/00-raw/atp_matches_2024.csv")
# ============================
# Data Wrangling: Variables of Interest + Surface Representativeness
# ============================

# 1) Quick preview (shows the dataset actually loaded correctly)
print("Dataset shape:", df.shape)
display(df.head(3))

# 2) Define the variables of interest for the hypothesis
required_cols = [
    "surface",
    "w_svpt", "w_1stIn", "w_1stWon", "w_2ndWon",   # winner serve stats
    "l_svpt", "l_1stIn", "l_1stWon", "l_2ndWon",   # loser serve stats
    "score"
]

missing_required = [c for c in required_cols if c not in df.columns]
if missing_required:
    print("Missing expected columns:", missing_required)
    raise KeyError("Some expected columns for serve/surface analysis are missing.")

# 3) Show sample rows for hypothesis-related columns
cols_of_interest = [
    "tourney_name","tourney_date","surface",
    "winner_name","loser_name",
    "w_svpt","w_1stIn","w_1stWon","w_2ndWon",
    "l_svpt","l_1stIn","l_1stWon","l_2ndWon",
    "score","minutes"
]
cols_of_interest = [c for c in cols_of_interest if c in df.columns]
print("\nSample rows for variables used in the hypothesis:")
display(df[cols_of_interest].head(8))

# 4) Surface distribution check (representativeness)
surface_counts = df["surface"].value_counts(dropna=False)
surface_perc = df["surface"].value_counts(dropna=False, normalize=True) * 100
surface_dist = pd.DataFrame({"count": surface_counts, "percent": surface_perc.round(2)})
print("\nCourt surface distribution (counts + percents):")
display(surface_dist)

ax = surface_counts.dropna().plot(kind="bar", title="Match Counts by Court Surface")
ax.set_xlabel("Surface")
ax.set_ylabel("Number of matches")
plt.tight_layout()
plt.show()

# 5) Missingness for variables of interest
interest_cols = ["surface","w_svpt","w_1stIn","w_1stWon","w_2ndWon","l_svpt","l_1stIn","l_1stWon","l_2ndWon"]
missing_interest = df[interest_cols].isna().sum().to_frame("missing_count")
missing_interest["missing_percent"] = (missing_interest["missing_count"] / len(df) * 100).round(2)
print("\nMissingness for variables of interest:")
display(missing_interest)

print("\nBasic summary stats (raw counts) for serve-related columns:")
display(df[interest_cols[1:]].describe().T)

# 6) Clean/tidy subset for analysis
#    - Remove incomplete matches (RET/W/O) because serve stats can be unreliable/missing
#    - Require non-null surfaces and denominators > 0
analysis_df = df.copy()

analysis_df["is_ret_wo"] = analysis_df["score"].astype(str).str.contains(r"RET|W/O", regex=True, na=False)

# Define denominators for % calculations
analysis_df["w_2ndIn"] = analysis_df["w_svpt"] - analysis_df["w_1stIn"]
analysis_df["l_2ndIn"] = analysis_df["l_svpt"] - analysis_df["l_1stIn"]

# Compute serve win percentages
analysis_df["w_1st_win_pct"] = analysis_df["w_1stWon"] / analysis_df["w_1stIn"]
analysis_df["w_2nd_win_pct"] = analysis_df["w_2ndWon"] / analysis_df["w_2ndIn"]
analysis_df["l_1st_win_pct"] = analysis_df["l_1stWon"] / analysis_df["l_1stIn"]
analysis_df["l_2nd_win_pct"] = analysis_df["l_2ndWon"] / analysis_df["l_2ndIn"]

# Keep only valid rows
pct_cols = ["w_1st_win_pct","w_2nd_win_pct","l_1st_win_pct","l_2nd_win_pct"]
clean_mask = (
    analysis_df["surface"].notna() &
    (~analysis_df["is_ret_wo"]) &
    (analysis_df["w_1stIn"] > 0) &
    (analysis_df["w_2ndIn"] > 0) &
    (analysis_df["l_1stIn"] > 0) &
    (analysis_df["l_2ndIn"] > 0)
)

analysis_df = analysis_df.loc[clean_mask].dropna(subset=pct_cols).copy()

# Sanity check: percentages should be within [0,1]
for c in pct_cols:
    analysis_df = analysis_df[(analysis_df[c] >= 0) & (analysis_df[c] <= 1)]

print("\nAfter cleaning:")
print("Clean analysis_df shape:", analysis_df.shape)
print("Surfaces in cleaned data:")
display(analysis_df["surface"].value_counts().to_frame("count"))

# 7) Summary stats for the percentages
print("\nOverall summary stats for serve win % variables:")
display(analysis_df[pct_cols].describe().T)

# 8) Save the cleaned analysis dataset
analysis_path = "data/02-processed/atp_matches_2024_analysis.csv"
try:
    from pathlib import Path
    Path("data/02-processed").mkdir(parents=True, exist_ok=True)
    analysis_df.to_csv(analysis_path, index=False)
    print("\nSaved cleaned analysis dataset to:", analysis_path)
except Exception as e:
    print("\n(Info) Could not save to disk in this environment:", e)

## Exploratory Data Analysis

In the data wrangling section above, we loaded the full ATP 2024 match dataset (3,076 matches), removed incomplete matches (retirements and walkovers), computed first-serve and second-serve win percentages, and filtered out rows with missing serve statistics. This produced a cleaned dataset of complete matches with valid serve metrics.

For the EDA, we focus on **2024 Grand Slam matches only**, consistent with our research question. We reshape each match into two **player-level observations** (one for the winner and one for the loser) so that every player's serve performance in each match is analyzed independently. This means a match between Player A and Player B produces two rows — one for each player's serve statistics — giving us approximately twice as many observations as matches. This reshaping is done transparently in the code cell below.

In [ ]:
import seaborn as sns

# Set plotting style
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['figure.dpi'] = 120
plt.rcParams['figure.figsize'] = (10, 6)

# Filter to Grand Slam matches only (tourney_level == 'G')
grand_slam_df = analysis_df[analysis_df['tourney_level'] == 'G'].copy()
print(f"Grand Slam matches after cleaning: {len(grand_slam_df)}")
print(f"\nTournament breakdown:")
display(grand_slam_df['tourney_name'].value_counts().to_frame('matches'))

# Reshape to player-level observations
# Each match produces two rows: one for the winner's serve stats, one for the loser's.
# This allows us to analyze every player's serve performance independently of match outcome.

winner_stats = grand_slam_df[['surface', 'tourney_name',
                               'w_1st_win_pct', 'w_2nd_win_pct',
                               'w_ace', 'w_df', 'w_svpt', 'w_1stIn']].copy()
winner_stats.columns = ['surface', 'tournament',
                         'first_serve_win_pct', 'second_serve_win_pct',
                         'aces', 'double_faults', 'serve_points', 'first_serves_in']
winner_stats['outcome'] = 'Winner'

loser_stats = grand_slam_df[['surface', 'tourney_name',
                              'l_1st_win_pct', 'l_2nd_win_pct',
                              'l_ace', 'l_df', 'l_svpt', 'l_1stIn']].copy()
loser_stats.columns = ['surface', 'tournament',
                        'first_serve_win_pct', 'second_serve_win_pct',
                        'aces', 'double_faults', 'serve_points', 'first_serves_in']
loser_stats['outcome'] = 'Loser'

eda_df = pd.concat([winner_stats, loser_stats], ignore_index=True).dropna()

# Compute normalized rates (aces and DFs per serve point) to control for match length
eda_df['ace_rate'] = eda_df['aces'] / eda_df['serve_points']
eda_df['df_rate'] = eda_df['double_faults'] / eda_df['serve_points']

print(f"\nTotal Grand Slam player-performances analyzed: {len(eda_df)}")
print(f"\nBreakdown by surface:")
display(eda_df['surface'].value_counts().to_frame('player_performances'))

### Distribution of Serve Win Percentages

Before comparing serve performance across surfaces, we first examine the overall distributions of first-serve and second-serve win percentages to understand the general spread and central tendency of these metrics.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# First-serve win% distribution
axes[0].hist(eda_df['first_serve_win_pct'] * 100, bins=30, edgecolor='black', alpha=0.7, color='steelblue')
axes[0].axvline(eda_df['first_serve_win_pct'].mean() * 100, color='red', linestyle='--', 
                label=f"Mean: {eda_df['first_serve_win_pct'].mean()*100:.1f}%")
axes[0].axvline(eda_df['first_serve_win_pct'].median() * 100, color='orange', linestyle='--', 
                label=f"Median: {eda_df['first_serve_win_pct'].median()*100:.1f}%")
axes[0].set_title('Distribution of First-Serve Win %', fontweight='bold', fontsize=13)
axes[0].set_xlabel('First-Serve Win %')
axes[0].set_ylabel('Frequency')
axes[0].legend()

# Second-serve win% distribution
axes[1].hist(eda_df['second_serve_win_pct'] * 100, bins=30, edgecolor='black', alpha=0.7, color='coral')
axes[1].axvline(eda_df['second_serve_win_pct'].mean() * 100, color='red', linestyle='--', 
                label=f"Mean: {eda_df['second_serve_win_pct'].mean()*100:.1f}%")
axes[1].axvline(eda_df['second_serve_win_pct'].median() * 100, color='orange', linestyle='--', 
                label=f"Median: {eda_df['second_serve_win_pct'].median()*100:.1f}%")
axes[1].set_title('Distribution of Second-Serve Win %', fontweight='bold', fontsize=13)
axes[1].set_xlabel('Second-Serve Win %')
axes[1].set_ylabel('Frequency')
axes[1].legend()

plt.tight_layout()
plt.show()

The histograms show that first-serve win percentages are centered around 70-75%, while second-serve win percentages cluster around 48-55%. First-serve win percentages have a tighter distribution, reflecting the inherent advantage of a well-placed first serve. Second-serve win percentages are lower on average and more spread out, consistent with the fact that second serves are slower and give the returner more opportunity. These baseline distributions set the context for examining how surface type may shift these patterns.

### Serve Win Percentage by Court Surface

Our central question asks whether court surface affects first-serve and second-serve win percentages differently. We compare both metrics across clay, grass, and hard courts using boxplots and summary statistics.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
surface_order = ['Clay', 'Grass', 'Hard']
palette = {'Clay': '#D2691E', 'Grass': '#228B22', 'Hard': '#4169E1'}

# First-serve win% by surface
sns.boxplot(ax=axes[0], x='surface', y='first_serve_win_pct', data=eda_df,
            order=surface_order, hue='surface', palette=palette, legend=False)
axes[0].set_title('First-Serve Win % by Surface', fontweight='bold', fontsize=13)
axes[0].set_xlabel('Court Surface')
axes[0].set_ylabel('First-Serve Win %')
axes[0].set_ylim(0.3, 1.0)

# Second-serve win% by surface
sns.boxplot(ax=axes[1], x='surface', y='second_serve_win_pct', data=eda_df,
            order=surface_order, hue='surface', palette=palette, legend=False)
axes[1].set_title('Second-Serve Win % by Surface', fontweight='bold', fontsize=13)
axes[1].set_xlabel('Court Surface')
axes[1].set_ylabel('Second-Serve Win %')
axes[1].set_ylim(0.1, 0.85)

plt.tight_layout()
plt.show()

# Summary statistics table
print("Summary Statistics: Serve Win % by Surface\n")
summary = eda_df.groupby('surface')[['first_serve_win_pct', 'second_serve_win_pct']].agg(
    ['count', 'mean', 'median', 'std']
).round(4)
display(summary)

The boxplots suggest that first-serve win percentage varies noticeably across surfaces. Grass courts appear to have the highest median first-serve win percentage, followed by hard courts, with clay showing the lowest. This is consistent with the hypothesis that faster surfaces (grass) reward aggressive first serves more than slower surfaces (clay), where the ball bounces higher and gives returners more time to react.

For second-serve win percentage, the differences across surfaces appear smaller. The medians are relatively close, though clay appears slightly lower. This pattern is consistent with our hypothesis that second-serve performance is more stable across surfaces, since second serves rely more on placement and spin than on raw speed — attributes less affected by court pace.

**Note:** These are descriptive observations, not formal statistical tests. Words like "higher" and "lower" refer to observed averages, not statistically confirmed differences.

### Relationship Between First-Serve and Second-Serve Win Percentage

To explore whether a player who performs well on first serve also performs well on second serve, we examine the correlation between the two metrics, colored by surface. This helps assess whether surface affects both serve types in the same direction or differently.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))

for surface, color in palette.items():
    mask = eda_df['surface'] == surface
    ax.scatter(eda_df.loc[mask, 'first_serve_win_pct'] * 100, 
               eda_df.loc[mask, 'second_serve_win_pct'] * 100,
               alpha=0.4, label=surface, color=color, edgecolors='white', linewidth=0.3, s=40)

ax.set_xlabel('First-Serve Win %', fontsize=12)
ax.set_ylabel('Second-Serve Win %', fontsize=12)
ax.set_title('First-Serve vs Second-Serve Win % by Surface', fontweight='bold', fontsize=13)
ax.legend(title='Surface')

# Add overall correlation
corr = eda_df[['first_serve_win_pct', 'second_serve_win_pct']].corr().iloc[0, 1]
ax.annotate(f'Overall r = {corr:.3f}', xy=(0.05, 0.95), xycoords='axes fraction', fontsize=11,
            bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow', alpha=0.8))

plt.tight_layout()
plt.show()

The scatter plot shows a moderate positive correlation between first-serve and second-serve win percentages — players who win more points on first serve also tend to win more on second serve. However, the relationship is not extremely tight, suggesting that the two metrics capture partially different aspects of serve effectiveness.

The surface clusters overlap substantially, but grass-court observations (green) tend to sit higher on first-serve win percentage, while clay-court observations (brown) tend to sit lower. The vertical spread (second-serve win %) is more similar across surfaces, supporting the hypothesis that surface has a larger effect on first-serve outcomes than on second-serve outcomes.

### Supporting Context: Ace and Double Fault Rates by Surface

As supporting evidence, we examine ace rates (aces per serve point) and double fault rates by surface. Using rates rather than raw counts normalizes for match length. Faster surfaces are expected to produce higher ace rates due to reduced ball deceleration on the bounce. We also compare the four individual Grand Slam tournaments to check whether the two hard-court slams (Australian Open and US Open) show similar patterns.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Ace rate by surface
sns.boxplot(ax=axes[0], x='surface', y='ace_rate', data=eda_df,
            order=surface_order, hue='surface', palette=palette, legend=False)
axes[0].set_title('Ace Rate (per Serve Point) by Surface', fontweight='bold', fontsize=12)
axes[0].set_xlabel('Court Surface')
axes[0].set_ylabel('Ace Rate')

# Double fault rate by surface
sns.boxplot(ax=axes[1], x='surface', y='df_rate', data=eda_df,
            order=surface_order, hue='surface', palette=palette, legend=False)
axes[1].set_title('Double Fault Rate (per Serve Point) by Surface', fontweight='bold', fontsize=12)
axes[1].set_xlabel('Court Surface')
axes[1].set_ylabel('Double Fault Rate')

# Per-tournament comparison of first-serve win%
tourney_order = sorted(eda_df['tournament'].unique())
sns.boxplot(ax=axes[2], x='tournament', y='first_serve_win_pct', data=eda_df,
            order=tourney_order, hue='tournament', legend=False)
axes[2].set_title('First-Serve Win % by Tournament', fontweight='bold', fontsize=12)
axes[2].set_xlabel('Tournament')
axes[2].set_ylabel('First-Serve Win %')
axes[2].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.show()

# Tournament-level summary
print("\nServe statistics by tournament:\n")
tourney_summary = eda_df.groupby('tournament').agg(
    surface=('surface', 'first'),
    count=('first_serve_win_pct', 'count'),
    avg_1st_serve_win=('first_serve_win_pct', 'mean'),
    avg_2nd_serve_win=('second_serve_win_pct', 'mean'),
    avg_ace_rate=('ace_rate', 'mean'),
    avg_df_rate=('df_rate', 'mean')
).round(4)
display(tourney_summary)

Ace rates appear highest on grass courts, consistent with the expectation that faster surfaces produce more unreturnable serves. Clay courts show the lowest ace rates, reflecting the slower bounce that gives returners more time. Double fault rates appear relatively similar across surfaces, suggesting that the surface primarily affects how effectively serves convert into points rather than the rate of serving errors.

The per-tournament breakdown shows that the two hard-court Grand Slams (Australian Open and US Open) have broadly similar serve win patterns, though slight differences may exist due to conditions like altitude, ball type, and weather. This provides some reassurance that the surface grouping is meaningful, though we acknowledge that with only one or two tournaments per surface, tournament-specific effects cannot be fully separated from surface effects.

### EDA Summary

Our exploratory analysis reveals patterns consistent with the hypothesis that court surface influences serve performance:

- **First-serve win percentage** appears highest on grass and lowest on clay, with hard courts in between. This aligns with the idea that faster surfaces amplify the advantage of a powerful first serve.
- **Second-serve win percentage** shows smaller differences across surfaces, suggesting that second-serve effectiveness is more stable since it depends more on spin and placement than raw speed.
- **Ace rates** follow a similar pattern to first-serve win percentage, providing supporting evidence that surface pace affects serve outcomes.
- The **scatter plot** confirms that first-serve and second-serve performance are moderately correlated, but surface appears to shift first-serve outcomes more than second-serve outcomes.

Because Grand Slam surface is closely tied to tournament identity (Grass = Wimbledon, Clay = Roland Garros, Hard = Australian Open + US Open), these observed patterns may reflect both surface effects and tournament-specific conditions such as weather, ball type, and draw composition. These are descriptive findings based on 2024 Grand Slam data. Formal hypothesis testing in later stages of the project will help determine whether the observed differences are statistically meaningful.

## Ethics

Instructions: Keep the contents of this cell. For each item on the checklist
-  put an X there if you've considered the item
-  IF THE ITEM IS RELEVANT place a short paragraph after the checklist item discussing the issue.
  
Items on this checklist are meant to provoke discussion among good-faith actors who take their ethical responsibilities seriously. Your teams will document these discussions and decisions for posterity using this section.  You don't have to solve these problems, you just have to acknowledge any potential harm no matter how unlikely.

Here is a [list of real world examples](https://deon.drivendata.org/examples/) for each item in the checklist that can refer to.

[![Deon badge](https://img.shields.io/badge/ethics%20checklist-deon-brightgreen.svg?style=popout-square)](http://deon.drivendata.org/)

### A. Data Collection
 - [X] **A.1 Informed consent**: If there are human subjects, have they given informed consent, where subjects affirmatively opt-in and have a clear understanding of the data uses to which they consent?

> This project uses publicly available professional tennis match statistics, so no direct human subjects are involved and informed consent is not required.

 - [X] **A.2 Collection bias**: Have we considered sources of bias that could be introduced during data collection and survey design and taken steps to mitigate those?

> Our data only includes men’s singles Grand Slam main-draw matches, which may bias results toward elite players and exclude lower-ranked or qualifying-round competitors.
> 
 - [X] **A.3 Limit PII exposure**: Have we considered ways to minimize exposure of personally identifiable information (PII) for example through anonymization or not collecting information that isn't relevant for analysis?

> We only use publicly reported match-level performance statistics and player identifiers already in the public domain, without collecting any additional personal or sensitive information.

 - [ ] **A.4 Downstream bias mitigation**: Have we considered ways to enable testing downstream results for biased outcomes (e.g., collecting data on protected group status like race or gender)?

### B. Data Storage
 - [X] **B.1 Data security**: Do we have a plan to protect and secure data (e.g., encryption at rest and in transit, access controls on internal users and third parties, access logs, and up-to-date software)?

> All datasets are stored locally in course-restricted environments and consist solely of publicly available sports statistics, minimizing security and privacy risks.

 - [ ] **B.2 Right to be forgotten**: Do we have a mechanism through which an individual can request their personal information be removed?
 - [X] **B.3 Data retention plan**: Is there a schedule or plan to delete the data after it is no longer needed?
       
> The data will be retained only for the duration of the course project and deleted afterward since it is not needed beyond the academic analysis.

### C. Analysis
 - [X] **C.1 Missing perspectives**: Have we sought to address blindspots in the analysis through engagement with relevant stakeholders (e.g., checking assumptions and discussing implications with affected communities and subject matter experts)?
       
> Our analysis focuses on quantitative match outcomes and does not incorporate player, coaching, or contextual perspectives, which may limit interpretation of causal mechanisms.

 - [X] **C.2 Dataset bias**: Have we examined the data for possible sources of bias and taken steps to mitigate or address these biases (e.g., stereotype perpetuation, confirmation bias, imbalanced classes, or omitted confounding variables)?

> We acknowledge potential biases arising from surface-specific tournament conditions, player specialization, and unequal match counts across surfaces and years.
 
 - [X] **C.3 Honest representation**: Are our visualizations, summary statistics, and reports designed to honestly represent the underlying data?

> Visualizations and summary statistics are designed to accurately reflect observed serve win percentages without exaggerating surface differences.

 - [X] **C.4 Privacy in analysis**: Have we ensured that data with PII are not used or displayed unless necessary for the analysis?

> No private or sensitive information is used or displayed in the analysis beyond public professional match statistics.

 - [X] **C.5 Auditability**: Is the process of generating the analysis well documented and reproducible if we discover issues in the future?

> All data cleaning and analysis steps are documented in reproducible notebooks to allow future review and verification of results.

### D. Modeling
 - [ ] **D.1 Proxy discrimination**: Have we ensured that the model does not rely on variables or proxies for variables that are unfairly discriminatory?
 - [ ] **D.2 Fairness across groups**: Have we tested model results for fairness with respect to different affected groups (e.g., tested for disparate error rates)?
 - [X] **D.3 Metric selection**: Have we considered the effects of optimizing for our defined metrics and considered additional metrics?

> We carefully select first-serve and second-serve win percentages as core metrics, recognizing they capture serve effectiveness but not all aspects of match performance.

 - [ ] **D.4 Explainability**: Can we explain in understandable terms a decision the model made in cases where a justification is needed?
 - [X] **D.5 Communicate limitations**: Have we communicated the shortcomings, limitations, and biases of the model to relevant stakeholders in ways that can be generally understood?

> We explicitly communicate that our findings are correlational and limited to recent men’s Grand Slam matches, not causal or universally generalizable.

### E. Deployment
 - [ ] **E.1 Monitoring and evaluation**: Do we have a clear plan to monitor the model and its impacts after it is deployed (e.g., performance monitoring, regular audit of sample predictions, human review of high-stakes decisions, reviewing downstream impacts of errors or low-confidence decisions, testing for concept drift)?
 - [ ] **E.2 Redress**: Have we discussed with our organization a plan for response if users are harmed by the results (e.g., how does the data science team evaluate these cases and update analysis and models to prevent future harm)?
 - [ ] **E.3 Roll back**: Is there a way to turn off or roll back the model in production if necessary?
 - [ ] **E.4 Unintended use**: Have we taken steps to identify and prevent unintended uses and abuse of the model and do we have a plan to monitor these once the model is deployed?

## Team Expectations 

Team Expectations

Communication & Response Time: We will use WeChat for daily updates and GitHub for technical work. We expect everyone to reply to messages within 12 hours usually, and faster if a deadline is close.

Meetings: We will meet once a week (in-person or virtual). If online, we keep cameras on to stay engaged. If a member can't make it, they need to tell the group 24 hours in advance and post an update so the team isn't blocked.

Decision Making: We aim for consensus. If we disagree on a technical choice, we will vote. If it's a tie or a member isn't responding during a deadline rush (after 6 hours), the active members will make the decision to keep the project moving.

Fair Contribution: To get an "A", everyone must work on all parts of the project (data, analysis, coding, and writing). No one will just "write the report" or just "write the code." We will review each other's work to ensure everyone understands the whole project.

Managing Tasks: We track everything on GitHub Issues. A task isn't "started" until it's assigned on GitHub, and it's not "done" until the Pull Request is reviewed and merged.

Getting Stuck (The 2.5-Hour Rule): If a member is stuck on a problem for more than 2.5 hour, they must tell the team on WeChat immediately instead of waiting for the next meeting. We encourage asking for help early rather than hiding the issue until the deadline.

Tone & Conflict: We will use polite tone. We critique the code/ideas, not the person. If we have a conflict, we will get on a call to resolve it instead of arguing over text.

Accountability: If a member misses a deadline or "ghosts" the team, we will send a written warning requiring a specific fix within 1 week. If they still don't contribute, we will escalate the issue to the professor by Week 7.

## Project Timeline Proposal

| Meeting Date | Meeting Time | Completed Before Meeting | Discuss at Meeting |
|---|---|---|---|
| 3/10 | Wed 5 PM | Complete core analysis/modeling; Draft "Results" and "Discussion" sections. | Code review of analysis; Peer edit writing; Discuss "Ethics" and "Conclusion" sections. |
| 3/15 | Sun 5 PM | Complete full draft of Final Project Notebook; Run "Restart & Run All" to check for errors. | Final polish; Check for typos/grammar; Record Video Presentation (if required); Submit Final Project (Due 3/18). |
| 3/20 | Before 11:59 PM | NA | Turn in final project & course survey |